# LightGBM (Default Params) — F1 Race Position Prediction

Trains a `LGBMRegressor` with default hyperparameters and evaluates it with **temporal cross-validation** — an expanding-window walk-forward split over seasons, so every fold is only ever validated on a season that comes *after* the seasons it was trained on. This mirrors the real deployment setting (predict an upcoming season using only past seasons) and avoids the leakage a random/shuffled K-fold split would introduce.

Metrics (same as the baseline and random forest notebooks, for direct comparison):
- **MAE** (Mean Absolute Error) — average position error
- **Spearman ρ** — rank-order correlation between predicted and actual finishing positions
- **Macro F1** — treats each position (1–20) as a class; averages F1 equally across all positions regardless of frequency

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, f1_score
from scipy.stats import spearmanr

sys.path.insert(0, str(Path.cwd().parent.parent))
from pipeline.feature_engineering.feature_engineering import FINAL_FEATURES, TARGET

## 2. Load Data

Train/test splits were produced by the feature engineering pipeline and persisted as Parquet files under `data/`. `train.parquet` holds seasons 2019–2024; `test.parquet` holds the held-out 2025 season.

In [2]:
train = pd.read_parquet('../../data/train.parquet')
test = pd.read_parquet('../../data/test.parquet')

print(f"train: {train.shape[0]} rows, years {sorted(train['year'].unique())}")
print(f"test:  {test.shape[0]} rows, years {sorted(test['year'].unique())}")

train: 2556 rows, years [np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]
test:  479 rows, years [np.int32(2025)]


## 3. Prepare Features and Target

`RacePosition` (1–20) is the regression target. Model inputs are restricted to `FINAL_FEATURES` — the locked feature list from the feature engineering pipeline — which excludes identifier columns (`DriverId`, `DriverNumber`, `year`) that carry no predictive signal on their own.

In [3]:
train_X = train[FINAL_FEATURES]
train_Y = train[TARGET]
test_X = test[FINAL_FEATURES]
test_Y = test[TARGET]

## 4. Validate Data

Confirm no missing values before training — the feature engineering pipeline should have handled imputation upstream.

In [4]:
for name, df in [("train_X", train_X), ("train_Y", train_Y), ("test_X", test_X), ("test_Y", test_Y)]:
    n_null = df.isna().sum().sum() if hasattr(df, "columns") else df.isna().sum()
    status = "OK" if n_null == 0 else f"WARNING: {n_null} nulls"
    print(f"{name}: {status}")

train_X: OK
train_Y: OK
test_X: OK
test_Y: OK


## 5. Metric Helpers

Same definitions as the baseline and random forest notebooks, so scores are directly comparable.

In [5]:
def macro_f1(y_true, y_pred):
    """Round continuous predictions to nearest integer position, then compute macro F1."""
    y_pred_int = pd.Series(y_pred).round().clip(1, 20).astype(int)
    return f1_score(y_true.astype(int), y_pred_int, average="macro", zero_division=0)

def safe_spearmanr(y_true, y_pred):
    """Return 0.0 when predictions are constant (ρ is undefined for constant input)."""
    if pd.Series(y_pred).nunique() == 1:
        return 0.0
    return spearmanr(y_true, y_pred).statistic

def score(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "Spearman_rho": safe_spearmanr(y_true, y_pred),
        "Macro_F1": macro_f1(y_true, y_pred),
    }

## 6. Temporal Cross-Validation

Expanding-window walk-forward split over the 6 training seasons (2019–2024): fold *i* trains on every season strictly before the validation season and validates on the next one. This yields 5 folds:

| Fold | Train years | Validation year |
|---|---|---|
| 1 | 2019 | 2020 |
| 2 | 2019–2020 | 2021 |
| 3 | 2019–2021 | 2022 |
| 4 | 2019–2022 | 2023 |
| 5 | 2019–2023 | 2024 |

`LGBMRegressor` is trained with default hyperparameters (only `random_state` is fixed, for reproducibility, and `verbose=-1` to silence LightGBM's training logs) in every fold.

In [6]:
years = sorted(train['year'].unique())
cv_rows = []

for i in range(1, len(years)):
    train_years = years[:i]
    val_year = years[i]

    fold_train_mask = train['year'].isin(train_years)
    fold_val_mask = train['year'] == val_year

    fold_model = LGBMRegressor(random_state=42, verbose=-1)
    fold_model.fit(train_X[fold_train_mask], train_Y[fold_train_mask])
    fold_preds = fold_model.predict(train_X[fold_val_mask])

    metrics = score(train_Y[fold_val_mask], fold_preds)
    metrics["fold"] = i
    metrics["train_years"] = f"{train_years[0]}-{train_years[-1]}" if len(train_years) > 1 else str(train_years[0])
    metrics["val_year"] = val_year
    cv_rows.append(metrics)

cv_results = pd.DataFrame(cv_rows)[["fold", "train_years", "val_year", "MAE", "Spearman_rho", "Macro_F1"]]
cv_results

,fold,train_years,val_year,MAE,Spearman_rho,Macro_F1
0,1,2019,2020,4.018426,0.506021,0.083806
1,2,2019-2020,2021,3.748084,0.568852,0.090464
2,3,2019-2021,2022,3.783869,0.540800,0.069471
3,4,2019-2022,2023,3.436395,0.613626,0.116530
4,5,2019-2023,2024,3.225506,0.694280,0.094158


## 7. CV Summary

In [7]:
cv_summary = cv_results[["MAE", "Spearman_rho", "Macro_F1"]].agg(["mean", "std"])
print(f"{'Metric':<14} {'Mean':>8}  {'Std':>8}")
print("-" * 34)
for col in ["MAE", "Spearman_rho", "Macro_F1"]:
    print(f"{col:<14} {cv_summary.loc['mean', col]:>8.3f}  {cv_summary.loc['std', col]:>8.3f}")

Metric             Mean       Std
----------------------------------
MAE               3.642     0.312
Spearman_rho      0.585     0.073
Macro_F1          0.091     0.017


## 8. Final Model — Evaluate on Held-Out 2025 Test Set

Refit on the full training set (2019–2024) and score once on the untouched 2025 test season, alongside the baselines and Random Forest results already saved in `reports/random_forest_test_results.csv` for comparison.

In [8]:
final_model = LGBMRegressor(random_state=42, verbose=-1)
final_model.fit(train_X, train_Y)
test_preds = final_model.predict(test_X)

test_results = score(test_Y, test_preds)

prior_results = pd.read_csv("../../reports/random_forest_test_results.csv").set_index("model")
comparison = pd.concat([
    prior_results,
    pd.DataFrame({"LightGBM": test_results}).T.rename_axis("model"),
])

print(f"{'Model':<20} {'MAE':>6}  {'Spearman ρ':>10}  {'Macro F1':>8}")
print("-" * 52)
for model, metrics in comparison.iterrows():
    print(f"{model:<20} {metrics['MAE']:>6.2f}  {metrics['Spearman_rho']:>10.3f}  {metrics['Macro_F1']:>8.3f}")

Model                   MAE  Spearman ρ  Macro F1
----------------------------------------------------
GridPosition          10.44       0.652     0.005
DummyRegressor         4.99       0.000     0.005
RandomForest           3.53       0.617     0.077
LightGBM               3.52       0.608     0.088


## 9. Train vs CV Error — Overfitting Check

Compare error on the data `final_model` was fit on (train, in-sample) against the mean CV error (out-of-sample, Section 7) and the held-out 2025 test error (Section 8). A large train-vs-CV/test gap signals overfitting — a real risk here since the model above uses unrestricted depth / unlimited boosting rounds by default.

In [9]:
train_preds = final_model.predict(train_X)
train_results = score(train_Y, train_preds)

error_comparison = pd.DataFrame({
    "Train (in-sample)": train_results,
    "CV (mean, out-of-sample)": cv_summary.loc["mean"],
    "Test (2025, held-out)": test_results,
}).T[["MAE", "Spearman_rho", "Macro_F1"]]

error_comparison

,MAE,Spearman_rho,Macro_F1
Train (in-sample),2.107542,0.886063,0.158878
"CV (mean, out-of-sample)",3.642456,0.584716,0.090886
"Test (2025, held-out)",3.520125,0.608012,0.087795


## 10. Feature Importance

First-pass feature importance from `final_model` (fit on the full 2019–2024 training set in Section 8). The scikit-learn API's default `importance_type='split'` counts how often a feature is used to split, not its impact on error — a different basis than Random Forest's impurity-based importance, so magnitudes aren't directly comparable across the two models, only the relative ranking within each.

In [10]:
feature_importance = pd.Series(
    final_model.feature_importances_, index=FINAL_FEATURES, name="importance"
).sort_values(ascending=False)

feature_importance

DriverFinish_ewm               471
TeamFinish_ewm                 455
LapStd_lag1                    447
TeamFinish_roll3_inseason      338
DriverFinish_roll3_inseason    270
GridPosition                   248
Meeting.Circuit.ShortName      211
round_number                   201
DriverFinish_lag1              197
TeamName                       162
Name: importance, dtype: int32

## 11. Save Results

In [11]:
cv_results.to_csv("../../reports/lightgbm_cv_results.csv", index=False)
comparison.reset_index().to_csv("../../reports/lightgbm_test_results.csv", index=False)
comparison

,MAE,Spearman_rho,Macro_F1
model,,,
GridPosition,10.437773,0.651649,0.005263
DummyRegressor,4.990574,0.000000,0.004771
RandomForest,3.531587,0.616917,0.077124
LightGBM,3.520125,0.608012,0.087795


## 12. Takeaways

- The temporal CV folds show how default-parameter LightGBM performance evolves as more historical seasons become available for training — early folds (fold 1: train on a single season) are a much weaker test than later folds.
- The final row compares the model, fit on all six training seasons, against the `GridPosition`/`DummyRegressor` baselines and `RandomForest` on the untouched 2025 season.
- **Overfitting is present but narrower than Random Forest**: train MAE is 2.11 vs. 3.64 on CV and 3.52 on test — a ~1.5-point gap, versus RandomForest's ~2.3-point gap (see its notebook, Section 9). LightGBM's boosted, shallower trees generalize somewhat better out of the box, though the gap is still large enough that `num_leaves`, `min_child_samples`, and early stopping on `n_estimators` are worth tuning.
- **Feature importance (split-count) is more evenly spread**: `DriverFinish_ewm`, `TeamFinish_ewm`, and `LapStd_lag1` lead, while `GridPosition` ranks 6th of 10 — a notable divergence from Random Forest, where `GridPosition` is the single dominant feature. LightGBM's boosting spreads splits across the engineered rolling/EWM form features rather than concentrating on qualifying position.
